In [1]:
import torch
import numpy as np


In [2]:
with open('/home/satvik/spark/spark/packet_seq.txt', 'r') as file:
    lines = file.readlines()

data = []
for line in lines:
    row = list(map(int, line.split()))
    data.append(row)

data_np = np.array(data)
data_np_reshaped = data_np.reshape(4, -1) 
data_tensor = torch.tensor(data_np_reshaped, dtype=torch.int)

print("Tensor shape:", data_tensor.shape)
print("Tensor data:\n", data_tensor)

Tensor shape: torch.Size([4, 469])
Tensor data:
 tensor([[  2, 111,   1,  ...,   0,   0,   0],
        [  2, 111,   1,  ...,   0,   0,   0],
        [  2, 111,   1,  ...,   0,   0,   0],
        [  2, 111,   1,  ...,  62,  22,   3]], dtype=torch.int32)


In [3]:
with open('/home/satvik/spark/spark/padded_field_pos.txt', 'r') as file:
    lines = file.readlines()

data_f = []
for line in lines:
    row = list(map(int, line.split()))
    data_f.append(row)

data_np_f = np.array(data_f)
data_np_reshaped_f = data_np_f.reshape(4, -1) 
data_tensor_f = torch.tensor(data_np_reshaped_f, dtype=torch.int)

print("Tensor shape:", data_tensor_f.shape)
print("Tensor data:\n", data_tensor_f)

Tensor shape: torch.Size([4, 469])
Tensor data:
 tensor([[ 1,  2,  3,  ...,  0,  0,  0],
        [ 1,  2,  3,  ...,  0,  0,  0],
        [ 1,  2,  3,  ...,  0,  0,  0],
        [ 1,  2,  3,  ..., 34,  0,  0]], dtype=torch.int32)


In [4]:
import random 

def apply_sfbo_masking(packet_seq, field_pos, max_span_length, padding_value=4):
    valid_indices_field = field_pos[field_pos != 0]
    valid_indices_packet = packet_seq[packet_seq != 0]
    # print(valid_indices_field)
    # print(valid_indices_packet)
    count = 0 
    unique_fields = torch.unique(valid_indices_field).tolist()
    # print(unique_fields)
    
    num_spans = random.randint(1, max_span_length)

    start_indices = []
    end_indices = []
    # span_indices = []

    if num_spans > len(unique_fields):
        raise ValueError("num_spans exceeds the length of unique_elements list")
    start_index = random.randint(0, len(unique_fields) - num_spans)
    span_selection = unique_fields[start_index:start_index + num_spans]

    span_positions = [i for i, value in enumerate(field_pos) if value in span_selection]
    masked_packet_seq = packet_seq.clone()
    masked_packet_seq[span_positions] = padding_value
    masked_packet_seq_tensor = masked_packet_seq.clone().detach()

    # start_indices.append(min(span_positions))
    # end_indices.append(max(span_positions))

    # start_indices_tensor = torch.tensor(start_indices)
    # end_indices_tensor = torch.tensor(end_indices)
    # span_indices_tensor = torch.tensor(span_positions)
    # # print(start_indices_tensor)
    # # print(end_indices_tensor)
    # # print(num_spans)
    # # print(span_selection)
    # # print(span_indices_tensor)
    # print(masked_packet_seq)

    count += len(span_positions)

    return masked_packet_seq_tensor

In [9]:
list_spans = []
count_spans = 0
for i in range(len(data_tensor)):
    p = data_tensor[i][:]
    f = data_tensor_f[i][:]
    print(len(p))
    # print(f)
    # print(p)
    x, c = apply_sfbo_masking(p, f, max_span_length=6)
    count_spans += c
    print("count: ", count_spans)
    list_spans.append(x)
    print(x.shape)

result_tensor = torch.stack(list_spans, dim=0)
print(result_tensor.shape)


469
count:  10
torch.Size([469])
469
count:  18
torch.Size([469])
469
count:  24
torch.Size([469])
469
count:  31
torch.Size([469])
torch.Size([4, 469])


In [8]:
result_tensor[0][:]

tensor([  2, 111,   1,   1, 105, 108,   1, 106,   1, 106,  29,  76,  90,   1,
         56, 137,  89,   1,  56, 134,   1, 120,   1,   1,  89,   1,  73,   1,
         45,   1,   1,   1,   1,   1,   1,   1,   4,   4,   1,   1,   1,   1,
          1,   1,   1,   1,   1,   1,   1,   1,   1,   1,   1, 159,   1,   1,
          1, 157,   1,   1,   1,   1,   1,   1,   3,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,  